# Week 2: Sentinel-2 Data Acquisition
### Lagos Barrier Island Shoreline Change Detection Project

Purpose: Query Google Earth Engine for the lowest-cloud-cover Sentinel-2 scene
per year (2017–2026) over the Lagos Barrier Island AOI, within the
November–March dry-season window.

In [1]:
import ee
import geemap

try:
    ee.Initialize()
except Exception:
    ee.Authenticate
    ee.Initialize


In [11]:
aoi = ee.Geometry.Rectangle([3.30, 6.38, 3.55, 6.48])

In [10]:
def get_best_scene(year, aoi, start_month=11, end_month=3, cloud_limit=20):
    """
    Returns the lowest-cloud-cover Sentinel-2 L2A image for the given
    dry-season window spanning (year) Nov through (year+1) March.
    """
    start_date = ee.Date.fromYMD(year, start_month, 1)
    end_date = ee.Date.fromYMD(year + 1, end_month, 1).advance(1, "month")

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_limit))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )

    count = collection.size().getInfo()
    if count == 0:
        print(f"No scenes found for {year}-{year+1} dry season "
              f"under {cloud_limit}% cloud cover.")
        return None

    best = collection.first()
    cloud_pct = best.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
    scene_date = best.date().format("YYYY-MM-dd").getInfo()
    print(f"{year}-{year+1}: selected scene from {scene_date} "
          f"({cloud_pct:.1f}% cloud, {count} candidates in window)")

    return best.clip(aoi)

In [16]:
years = list(range(2017, 2026))  # dry-season windows: 2016-17 ... 2025-26

selected_scenes = {}
for yr in years:
    scene = get_best_scene(yr, aoi)
    if scene is not None:
        selected_scenes[yr] = scene

print(f"\nCompleted scene selection: {len(selected_scenes)} / {len(years)} years have a usable scene.")

2017-2018: selected scene from 2018-01-06 (0.0% cloud, 2 candidates in window)
2018-2019: selected scene from 2019-01-06 (0.0% cloud, 7 candidates in window)
2019-2020: selected scene from 2020-01-26 (1.3% cloud, 11 candidates in window)
2020-2021: selected scene from 2021-01-20 (4.4% cloud, 5 candidates in window)
2021-2022: selected scene from 2022-01-25 (0.0% cloud, 4 candidates in window)
2022-2023: selected scene from 2022-12-26 (0.0% cloud, 5 candidates in window)
2023-2024: selected scene from 2023-12-16 (0.0% cloud, 6 candidates in window)
2024-2025: selected scene from 2024-12-20 (0.0% cloud, 11 candidates in window)
2025-2026: selected scene from 2026-01-24 (1.4% cloud, 1 candidates in window)

Completed scene selection: 9 / 9 years have a usable scene.


In [13]:
# One-off diagnostic: what's available for 2016-2017 at a looser threshold?
test = get_best_scene(2016, aoi, cloud_limit=40)

No scenes found for 2016-2017 dry season under 40% cloud cover.


In [14]:
# Diagnostic: is there ANY Sentinel-2 L2A data at all for this window?
raw_check = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(ee.Date.fromYMD(2016, 11, 1), ee.Date.fromYMD(2017, 4, 1))
)
print("Total images (no cloud filter):", raw_check.size().getInfo())

Total images (no cloud filter): 0


In [15]:
l1c_check = (
    ee.ImageCollection("COPERNICUS/S2_HARMONIZED")  # L1C collection
    .filterBounds(aoi)
    .filterDate(ee.Date.fromYMD(2016, 11, 1), ee.Date.fromYMD(2017, 4, 1))
)
print("Total L1C images:", l1c_check.size().getInfo())

Total L1C images: 18


Note on study period: The dataset begins in 2017 rather than 2016. Sentinel-2 L2A
(surface reflectance) coverage is not available for the Lagos AOI prior to March 2017,
confirmed by querying the `COPERNICUS/S2_SR_HARMONIZED` collection directly (zero images
returned for the Nov 2016–Mar 2017 window, independent of cloud filtering).

## Cloud Masking

Even though we selected the lowest-cloud scene per year, the cloud percentage
metadata is a scene-wide average — it doesn't guarantee our specific AOI is
clean. This step masks out cloud/shadow/snow pixels at the individual pixel
level, using Sentinel-2's built-in Scene Classification Layer (SCL) band,
before we compute anything on the imagery.

In [17]:
def mask_clouds_scl(image):
    """
    Masks out cloud, cloud shadow, and snow/ice pixels using the Sentinel-2
    Scene Classification Layer (SCL) band.

    SCL class values we exclude:
        3  = cloud shadow
        8  = cloud, medium probability
        9  = cloud, high probability
        10 = thin cirrus
        11 = snow/ice
    """
    scl = image.select("SCL")

    # Build a mask that is True (keep) only where SCL is NOT one of the
    # unwanted classes.
    mask = (
        scl.neq(3)
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    return image.updateMask(mask)

In [18]:
Map2 = geemap.Map(center=[6.43, 3.42], zoom=11)

vis_params = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}

# Pick one year to inspect closely as an example
sample_year = 2020
Map2.addLayer(selected_scenes[sample_year], vis_params, f"Unmasked {sample_year}")
Map2.addLayer(masked_scenes[sample_year], vis_params, f"Masked {sample_year}")
Map2.addLayer(aoi, {}, "AOI", opacity=0.3)
Map2

NameError: name 'masked_scenes' is not defined